# EMIT / Sentinel-2 — Spectral Regression (Color Matching)

Runs independently of `Pairs_Extract.ipynb`. Reads extracted tile pairs from Drive,
fits per-tile S2→EMIT spectral regression, produces synthetic 32-band imagery,
and saves per-tile R² results as CSV for downstream filtering.

**Inputs**: tile pairs on Drive (produced by `Pairs_Extract.ipynb`)  
**Outputs**: regression-synthetic GeoTIFFs, R² CSV, diagnostic plots

## 1. Setup

In [ ]:
import os, getpass
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

In [ ]:
import os, textwrap

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!pip install -q numpy scipy scikit-learn rasterio matplotlib tqdm joblib

In [ ]:
import json, glob
from pathlib import Path
from datetime import datetime
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from documentation.config import PipelineConfig

from spectral.s2_to_emit import (
    fit_tile, plot_r2_spectrum, plot_spectral_match,
)

from documentation.report_builder import (
    ReportBuilder, R2Aggregator,
    plot_r2_histogram, plot_r2_per_band,
)

print("All imports OK.")

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Parameters

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  USER PARAMETERS — edit these before running
# ═══════════════════════════════════════════════════════════════════════

DRIVE_ROOT = Path("/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches")
RUN_TAG    = "2026-03-22"   # must match the extraction run

# Spectral regression parameters (should match pipeline_defaults.json)
SPECTRAL_DEGREE = 2
SPECTRAL_ALPHA  = 1.0
SPECTRAL_MAX_CV = 0.25
EMIT_UPSAMPLE_ORDER = 1
TILE_SCALE = 6

SHOW_PLOTS = False   # set True for interactive exploration

# ═══════════════════════════════════════════════════════════════════════

In [ ]:
spectral_params = {
    "scale":              TILE_SCALE,
    "degree":             SPECTRAL_DEGREE,
    "alpha":              SPECTRAL_ALPHA,
    "max_cv":             SPECTRAL_MAX_CV,
    "emit_upsample_order": EMIT_UPSAMPLE_ORDER,
}

## 4. Discover tile pairs from Drive

Scans the Drive folder tree for `manifest.csv` files produced by `Pairs_Extract.ipynb`.
Each manifest lists the tile paths for one AOI/pair.

In [ ]:
drive_base = DRIVE_ROOT / RUN_TAG

# Discover all manifests: {drive_base}/{aoi_slug}/{pair_id}/manifest.csv
manifests = sorted(drive_base.glob("*/*/manifest.csv"))
print(f"Found {len(manifests)} pair manifest(s) under {drive_base}")
for m in manifests:
    print(f"  {m.parent.parent.name} / {m.parent.name}")

## 5. Run spectral regression for all pairs

For each pair: fit per-tile regression, produce synthetic GeoTIFFs,
and collect R² results.

In [ ]:
all_r2_rows = []  # collects per-tile R² for the global CSV

for manifest_path in tqdm(manifests, desc="Pairs"):
    pair_dir = manifest_path.parent
    pair_id  = pair_dir.name
    aoi_slug = pair_dir.parent.name

    # Read manifest
    mf = pd.read_csv(manifest_path)
    if "emit_b32_tif" not in mf.columns or "s2_tif" not in mf.columns:
        print(f"  {aoi_slug}/{pair_id}: manifest missing required columns — skipping")
        continue

    # Output directories
    synth_dir   = pair_dir / "regression_synth"
    matcher_dir = pair_dir / "regression_matchers"
    plots_dir   = pair_dir / "plots"
    synth_dir.mkdir(parents=True, exist_ok=True)
    matcher_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Read wavelengths from metadata
    wl_path = list(pair_dir.glob("metadata/*.wavelengths.json")) or \
              list(pair_dir.glob("**/*.wavelengths.json"))
    if wl_path:
        with open(wl_path[0]) as f:
            wl_nm = np.array(json.load(f))
    else:
        # Fallback: read from first EMIT tile
        import rasterio
        first_emit = mf["emit_b32_tif"].dropna().iloc[0]
        with rasterio.open(first_emit) as src:
            wl_nm = np.array([float(d.split("_")[0]) for d in src.descriptions if d])
        if len(wl_nm) == 0:
            print(f"  {aoi_slug}/{pair_id}: cannot determine wavelengths — skipping")
            continue

    # Determine b32 wavelengths from first tile
    first_b32 = mf["emit_b32_tif"].dropna().iloc[0]
    import rasterio
    with rasterio.open(first_b32) as src:
        n_bands_b32 = src.count
    # If wl_nm has 285 bands but tiles have 32, we need the b32 subset
    if len(wl_nm) > n_bands_b32:
        # Try to get indices from tile metadata
        meta_jsons = sorted(pair_dir.glob("metadata/tiles/*.json"))
        if meta_jsons:
            with open(meta_jsons[0]) as f:
                tile_meta = json.load(f)
            b32_idx = tile_meta.get("emit_b32_indices_0based")
            if b32_idx:
                wl_b32 = wl_nm[b32_idx]
            else:
                from data.EMIT.emit_utils import select_emit_bands
                _, b32_idx = select_emit_bands(wl_nm, num_keep=32)
                wl_b32 = wl_nm[b32_idx]
        else:
            from data.EMIT.emit_utils import select_emit_bands
            _, b32_idx = select_emit_bands(wl_nm, num_keep=32)
            wl_b32 = wl_nm[b32_idx]
    else:
        wl_b32 = wl_nm

    r2_agg = R2Aggregator(wavelengths_nm=wl_nm)
    n_fitted = 0

    for _, row in mf.iterrows():
        tile_idx   = int(row["idx"])
        s2_tif     = row["s2_tif"]
        b32_tif    = row.get("emit_b32_tif")

        if pd.isna(b32_tif) or pd.isna(s2_tif):
            continue
        if not Path(b32_tif).exists() or not Path(s2_tif).exists():
            print(f"    tile {tile_idx:03d}: files missing — skipping")
            continue

        # Check if regression output already exists
        out_name = Path(s2_tif).name.replace("_s2.tif", "_regression_synth.tif")
        out_path = synth_dir / out_name
        if out_path.exists():
            continue

        try:
            matcher, stats = fit_tile(
                s2_tile_path=s2_tif,
                emit_b32_tile_path=b32_tif,
                **spectral_params,
                verbose=False,
            )
        except ValueError as e:
            print(f"    tile {tile_idx:03d}: SKIPPED — {e}")
            continue

        # Save matcher
        matcher.save(matcher_dir / f"tile_{tile_idx:03d}_matcher.joblib")

        # Apply to produce synthetic tile
        matcher.apply_to_tile(s2_tif, out_path=out_path)

        # Collect R²
        r2_agg.add(tile_idx, np.array(stats["r2_per_band"]), stats["r2_mean"])
        all_r2_rows.append({
            "aoi_slug":    aoi_slug,
            "pair_id":     pair_id,
            "tile_idx":    tile_idx,
            "r2_mean":     stats["r2_mean"],
            **{f"r2_band_{i:02d}": v for i, v in enumerate(stats["r2_per_band"])},
        })
        n_fitted += 1

        # Optional diagnostic plot
        if SHOW_PLOTS or n_fitted <= 2:
            plot_spectral_match(
                s2_tif, str(out_path), b32_tif,
                title_suffix=f"tile {tile_idx:03d}",
                r2_mean=stats["r2_mean"],
                save_path=str(plots_dir / f"{pair_id}_tile{tile_idx:03d}_synth.png"),
                show=SHOW_PLOTS,
                wavelengths_nm=wl_b32,
            )

    # Per-pair summary
    if n_fitted > 0:
        r2_summary = r2_agg.summary()
        print(f"  {aoi_slug}/{pair_id}: {n_fitted} tiles, "
              f"R² mean={r2_summary['global_mean']:.4f}, "
              f"median={r2_summary['global_median']:.4f}")

        # Per-pair R² plots
        plot_r2_histogram(
            r2_agg.r2_mean,
            plots_dir / "r2_histogram.png",
            title=f"R² distribution — {pair_id}",
        )
        plot_r2_per_band(
            r2_agg.r2_per_band, r2_agg.wavelengths_nm,
            plots_dir / "r2_per_band.png",
        )
    else:
        print(f"  {aoi_slug}/{pair_id}: no tiles fitted")

print(f"\nDone. {len(all_r2_rows)} tiles processed across {len(manifests)} pairs.")

## 6. Save R² results

Writes a single CSV with per-tile R² (mean + per-band) across all AOIs/pairs.
This CSV is the input for downstream R²-based filtering in SR training.

In [ ]:
r2_df = pd.DataFrame(all_r2_rows)

r2_csv_path = drive_base / "r2_all_tiles.csv"
r2_df.to_csv(r2_csv_path, index=False)

print(f"R² CSV saved: {r2_csv_path}  ({len(r2_df)} tiles)")
print(f"\nR² mean summary:")
print(f"  Mean:   {r2_df['r2_mean'].mean():.4f}")
print(f"  Median: {r2_df['r2_mean'].median():.4f}")
print(f"  Min:    {r2_df['r2_mean'].min():.4f}")
print(f"  Max:    {r2_df['r2_mean'].max():.4f}")
print(f"  Std:    {r2_df['r2_mean'].std():.4f}")

r2_df.head(10)

## 7. Global R² diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of mean R² across all tiles
axes[0].hist(r2_df["r2_mean"], bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(r2_df["r2_mean"].median(), color="red", ls="--", label=f"median={r2_df['r2_mean'].median():.3f}")
axes[0].set_xlabel("Mean R²")
axes[0].set_ylabel("Count")
axes[0].set_title("R² distribution (all tiles)")
axes[0].legend()

# Per-AOI box plot
aoi_order = r2_df.groupby("aoi_slug")["r2_mean"].median().sort_values(ascending=False).index
r2_df_sorted = r2_df.set_index("aoi_slug").loc[aoi_order].reset_index()
r2_df_sorted.boxplot(column="r2_mean", by="aoi_slug", ax=axes[1], rot=90)
axes[1].set_title("R² by AOI")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean R²")
plt.suptitle("")

plt.tight_layout()
fig.savefig(str(drive_base / "r2_global_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTiles with R² >= 0.75: {(r2_df['r2_mean'] >= 0.75).sum()} / {len(r2_df)}")
print(f"Tiles with R² >= 0.80: {(r2_df['r2_mean'] >= 0.80).sum()} / {len(r2_df)}")